# PhishGuard — Exploratory Data Analysis

Author: Akshay | https://akshay.fruvvi.com

This notebook explores the PhishTank + Tranco datasets to:
- Understand class distribution
- Visualize feature distributions (phishing vs legitimate)
- Identify the most discriminative features
- Detect and handle outliers
- Validate the feature engineering pipeline

In [ ]:
import sys
sys.path.insert(0, '../training')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from features import extract, FEATURE_NAMES

plt.style.use('dark_background')
sns.set_palette('husl')
%matplotlib inline

print('Libraries loaded.')

## 1. Load Dataset

In [ ]:
datasets_dir = Path('../datasets')
frames = []

for path in datasets_dir.glob('phishing_*.csv'):
    df = pd.read_csv(path)
    df.columns = df.columns.str.lower()
    df['label'] = 1
    frames.append(df[['url', 'label']])
    print(f'Loaded {len(df):,} phishing URLs from {path.name}')

for path in datasets_dir.glob('legit_*.csv'):
    df = pd.read_csv(path)
    df.columns = df.columns.str.lower()
    df['label'] = 0
    frames.append(df[['url', 'label']])
    print(f'Loaded {len(df):,} legitimate URLs from {path.name}')

data = pd.concat(frames, ignore_index=True).dropna().drop_duplicates(subset=['url'])
print(f'\nTotal: {len(data):,} URLs')
data['label'].value_counts()

## 2. Extract Features

In [ ]:
print('Extracting features...')
feature_list = [extract(url).to_list() for url in data['url']]
features_df = pd.DataFrame(feature_list, columns=[
    'url_length', 'domain_length', 'subdomain_count', 'has_ip', 'no_https',
    'special_chars', 'digit_ratio', 'entropy', 'path_depth', 'has_login_kw',
    'has_brand_kw', 'suspicious_tld', 'has_redirect', 'domain_age', 'has_port'
])
features_df['label'] = data['label'].values
print(f'Feature matrix shape: {features_df.shape}')
features_df.head()

## 3. Feature Distributions by Class

In [ ]:
continuous_features = ['url_length', 'domain_length', 'subdomain_count', 
                       'special_chars', 'digit_ratio', 'entropy', 'path_depth']

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

phish = features_df[features_df['label'] == 1]
legit = features_df[features_df['label'] == 0]

for i, feat in enumerate(continuous_features):
    axes[i].hist(legit[feat], bins=50, alpha=0.6, label='Legitimate', color='#22c55e', density=True)
    axes[i].hist(phish[feat], bins=50, alpha=0.6, label='Phishing',   color='#ef4444', density=True)
    axes[i].set_title(feat, fontweight='bold')
    axes[i].legend(fontsize=8)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Density')

axes[-1].remove()
plt.suptitle('Feature Distributions: Phishing vs Legitimate', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../exports/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Correlation Matrix

In [ ]:
corr = features_df.drop('label', axis=1).corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../exports/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Phishing Rate by Feature Value

In [ ]:
binary_features = ['has_ip', 'no_https', 'has_login_kw', 'has_brand_kw', 
                   'suspicious_tld', 'has_redirect', 'has_port']

phish_rates = {}
for feat in binary_features:
    rate_when_true  = features_df[features_df[feat] > 0.5]['label'].mean()
    rate_when_false = features_df[features_df[feat] <= 0.5]['label'].mean()
    phish_rates[feat] = {'when_true': rate_when_true, 'when_false': rate_when_false}

rates_df = pd.DataFrame(phish_rates).T
print('Phishing rate by binary feature value:')
print(rates_df.sort_values('when_true', ascending=False).to_string(float_format='{:.1%}'.format))